In [ ]:
!pip -q install insightface onnxruntime opencv-python tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 86.7 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from collections import Counter
from sklearn.cluster import KMeans
from insightface.app import FaceAnalysis

Configuration

In [ ]:
WINDOW = 5                 # Rolling baseline window
MIN_DET = 0.6              # Face detection confidence
PERCENTILE_THRESH = 10     # Threshold percentile

Initialize Face Model

In [ ]:
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=-1, det_size=(640,640))

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:04<00:00, 59752.96KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


Utility Functions

In [ ]:
def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(np.dot(a, b))

def blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

Face Alignment

In [ ]:
ARC_TEMPLATE_112 = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041],
], dtype=np.float32)

def align_face(img, face, out_size=112):
    kps = face.kps.astype(np.float32)
    dst = ARC_TEMPLATE_112.copy()
    M, _ = cv2.estimateAffinePartial2D(kps, dst)
    if M is None:
        return None
    aligned = cv2.warpAffine(img, M, (out_size, out_size))
    return aligned

Quality Gate

In [ ]:
def is_good_capture(path, min_blur):
    img = cv2.imread(path)
    if img is None:
        return False, {"reason":"read_failed"}

    faces = app.get(img)
    if len(faces) != 1:
        return False, {"reason":"face_detect_fail"}

    f = faces[0]
    det = float(getattr(f, "det_score", 0))
    blur = blur_score(img)

    if det < MIN_DET:
        return False, {"det_score":det, "blur":blur}

    if blur < min_blur:
        return False, {"det_score":det, "blur":blur}

    return True, {"det_score":det, "blur":blur}

Build Embedding Bank

In [ ]:
def build_embedding_bank(folder):
    paths = sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    bank = []
    valid_paths = []

    for p in tqdm(paths):
        img = cv2.imread(p)
        if img is None:
            continue

        faces = app.get(img)
        if len(faces) != 1:
            continue

        f = faces[0]
        bank.append(f.normed_embedding)
        valid_paths.append(p)

    bank = np.vstack(bank)
    return valid_paths, bank

**Auto Threshold**

In [ ]:
def auto_threshold(paths, bank):
    sims = []
    for i in range(WINDOW, len(paths)):
        baseline = np.mean(bank[i-WINDOW:i], axis=0)
        today = bank[i]
        sims.append(cosine_sim(baseline, today))

    return float(np.percentile(sims, PERCENTILE_THRESH))

Rolling Baseline Decision

In [ ]:
def rolling_results(paths, bank, thresh, min_blur):
    results = []

    for i in range(WINDOW, len(paths)):
        today = paths[i]

        ok, info = is_good_capture(today, min_blur)
        if not ok:
            results.append({"status":"RETAKE", "info":info})
            continue

        baseline = np.mean(bank[i-WINDOW:i], axis=0)
        today_emb = bank[i]
        sim = cosine_sim(baseline, today_emb)

        if sim < thresh:
            results.append({"status":"POTENTIAL_CHANGE", "similarity":sim})
        else:
            results.append({"status":"OK", "similarity":sim})

    return results

Confirmation Rule

In [ ]:
def confirm_changes(results):
    confirmed = 0
    for i in range(1, len(results)):
        if results[i]["status"] == "POTENTIAL_CHANGE" and \
           results[i-1]["status"] == "POTENTIAL_CHANGE":
            confirmed += 1
    return confirmed

Emotion & Age (Phase-2 Placeholder)

In [ ]:
def emotion_detect(aligned_face):
    return "Neutral"

def age_estimate(aligned_face):
    return 25

Full Pipeline Runner

In [ ]:
def run_person(folder):

    print("Processing:", folder)

    paths, bank = build_embedding_bank(folder)

    # Auto threshold
    thresh = auto_threshold(paths, bank)
    print("Auto similarity threshold:", thresh)

    # Auto blur threshold (5th percentile)
    blur_vals = []
    for p in paths:
        img = cv2.imread(p)
        blur_vals.append(blur_score(img))

    min_blur = float(np.percentile(blur_vals, 5))
    print("Auto min_blur:", min_blur)

    results = rolling_results(paths, bank, thresh, min_blur)

    print("Summary:", Counter([r["status"] for r in results]))
    print("Confirmed changes:", confirm_changes(results))

    return results

In [ ]:
PERSON_A = "/content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/personA"
PERSON_B = "/content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/personB"

results_A = run_person(PERSON_A)
results_B = run_person(PERSON_B)

Processing: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/personA


100%|██████████| 44/44 [01:31<00:00,  2.09s/it]


Auto similarity threshold: 0.7175383567810059
Auto min_blur: 16.370113575531388
Summary: Counter({'OK': 34, 'RETAKE': 3, 'POTENTIAL_CHANGE': 2})
Confirmed changes: 0
Processing: /content/drive/MyDrive/Mirror Vision-KARIGOR/Dataset_Mirror_SPLIT/personB


100%|██████████| 93/93 [03:27<00:00,  2.23s/it]


Auto similarity threshold: 0.8705813944339752
Auto min_blur: 5.880373562633241
Summary: Counter({'OK': 78, 'POTENTIAL_CHANGE': 9, 'RETAKE': 1})
Confirmed changes: 2
